# Connect Google Sheets to GitHub Actions — V000001

**Goal:** connect the already-built JP publishing engine and verify its first live deployment. This notebook creates no Google Sheet, imports no replacement dataset, and performs no Drive inventory scan. Your V8 inventory notebook remains a separate tool; you do not need to rerun it to connect Actions.

## What you do

1. Use a Google Cloud project that you control. If you do not have one, [create a project](https://console.cloud.google.com/projectcreate), accept Google's terms if shown, and copy its **project ID**. Your Google account must be permitted to enable APIs and manage service accounts and workload identity pools in that project.
2. [Create a short-lived GitHub fine-grained token](https://github.com/settings/personal-access-tokens/new). Select **only JP_Communication-Command-Centre-and-Resource-Hub**, give **Actions: Read and write**, **Variables: Read and write**, and **Contents: Read-only**. Metadata read is automatic. Seven-day expiry is sufficient. Paste it only into the hidden prompt in this notebook, never into chat or a Sheet. Revoke it after setup.
3. Enter the project ID below, keep `CONNECT_ACTIONS = True`, then **Runtime → Run all**. Sign into the Google account that can share the existing source Sheets. The notebook asks for your token in a hidden input.

A service account is a machine identity, **not another Gmail account**. It uses keyless federation from the exact GitHub repository/owner IDs and publishing branch. No Google private key is created. Existing people keep their access. Source workbooks/original messages receive reader access; only the existing publishing workbook and its existing restore log receive writer access. Sharing notifications are disabled.

## Steps and checks

Configuration → registered-file preview → Google authorization → idempotent identity/permissions setup → verified repository variables → required-live Actions run → matching Pages receipt.

**Completion means** the exact Actions run succeeded and the deployed JSON names that run and commit with `live_google_sheets_fetch=true`. Prepared settings, skipped auth, an old successful run, and snapshot deployments do not count. If identity propagation or the job is still pending, the notebook reports pending and gives the run link. Rerun the final cell to resume safely; it reuses the same identities and file IDs.

The website publishes only reviewed/admitted public fields. Private attendance, payments and unreviewed evidence stay private. Automated Slack/Discord collection is not started. Schedule activation is recorded separately from identity setup.

## Version and restore

Artifact: **JP Actions Google Connection / V000001**. This is the first version of a different task from **JP Drive Inventory and Sheet History / V000008**. Subsequent changes must preserve this notebook's live Drive ID and archive its numbered PRE/POST versions in its own central history folder. No credential or executed authentication output may be saved in the distributed notebook.

In [ ]:
# Step 1 — enter the Google Cloud project ID (not its display name).
PROJECT_ID = ""  # @param {type:"string"}
CONNECT_ACTIONS = True  # @param {type:"boolean"}
WAIT_MINUTES = 25  # @param {type:"integer"}
# False runs an offline registered-file preview and cannot claim a connection.


## Engine
The next cell contains the same maintained helper as `scripts/connect_actions_google.py` in GitHub. It does not run setup merely by defining the functions. Tokens are never written to a file, printed or stored in cell source.

In [ ]:
#!/usr/bin/env python3
"""One-time, user-run Colab setup for the existing Sheets publishing engine.

Uses keyless federation. Never creates a service-account key or another Sheet.
Only exact registered files receive a service-account permission. Credentials
stay in memory; provider response bodies must never appear in notebook output.
"""
from __future__ import annotations

import copy
import re
import time
import uuid
from datetime import datetime, timezone
from urllib.parse import quote, urlparse

REPOSITORY = "FabinGurung/JP_Communication-Command-Centre-and-Resource-Hub"
BRANCH = "feature/daily-ops-current-work"
WORKFLOW = "daily-ops-branch-preview.yml"
CONFIG_PATH = "ops/control/sheets-actions-publishing.v1.json"
REGISTRY_PATH = "ops/control/branch-lane-registry.json"
SITE_ROOT = "https://fabingurung.github.io/JP_Communication-Command-Centre-and-Resource-Hub/"
POOL_ID = "jp-pages-github"
PROVIDER_ID = "github"
ACCOUNT_ID = "jp-pages-publisher"
ID_PATTERN = re.compile(r"[A-Za-z0-9_-]{10,200}")


class ConnectionError(RuntimeError):
    """Safe operational messages only; never include response text or tokens."""


def file_id(value):
    if not isinstance(value, str) or not ID_PATTERN.fullmatch(value):
        raise ConnectionError("REGISTERED_FILE_ID_INVALID")
    return value


def access_plan(cfg):
    """One entry per live object, irrespective of how many tabs consume it."""
    if cfg.get("repository") != REPOSITORY:
        raise ConnectionError("REPOSITORY_MISMATCH")
    plan = {}

    def add(fid, role, purpose):
        fid = file_id(fid)
        entry = plan.setdefault(fid, {"file_id": fid, "role": role, "purposes": []})
        if role != entry["role"]:
            raise ConnectionError("SOURCE_AND_OUTPUT_ID_COLLISION")
        if purpose not in entry["purposes"]:
            entry["purposes"].append(purpose)

    for name, spec in cfg["sources"].items():
        if spec.get("read_mode") != "EMPTY_TEMPLATE_NOT_CONNECTED":
            add(spec["workbook_id"], "reader", name)
    resource = cfg.get("resources_source")
    if resource:
        add(resource["workbook_id"], "reader", resource["tab"])
    for name, spec in cfg.get("notification_sources", {}).items():
        add(spec["workbook_id"], "reader", name)
    for spec in cfg.get("field_sources", []):
        add(spec["workbook_id"], "reader", spec["tab"])
    for original in cfg.get("notification_policy", {}).get("admitted_sources", {}).values():
        url = urlparse(original["drive_source_url"])
        match = re.fullmatch(r"/file/d/([A-Za-z0-9_-]+)/view", url.path)
        if url.scheme != "https" or url.netloc != "drive.google.com" or not match:
            raise ConnectionError("ORIGINAL_DRIVE_URL_INVALID")
        add(match.group(1), "reader", "Admitted original message; integrity verification")
    add(cfg["publishing_workbook_id"], "writer", "Derived publishing workbook only")
    add(cfg["previous_versions_file_id"], "writer", "Existing append-only publishing restore log")
    if cfg["previous_versions_folder_id"] in plan:
        raise ConnectionError("FOLDER_SHARING_FORBIDDEN")
    return list(plan.values())


def federation_spec(project_number, repo):
    rid, oid = str(repo.get("id", "")), str(repo.get("owner", {}).get("id", ""))
    if repo.get("full_name") != REPOSITORY or not rid.isdigit() or not oid.isdigit():
        raise ConnectionError("GITHUB_IMMUTABLE_ID_MISSING")
    if not str(project_number).isdigit():
        raise ConnectionError("GOOGLE_PROJECT_NUMBER_INVALID")
    pool = f"projects/{project_number}/locations/global/workloadIdentityPools/{POOL_ID}"
    return {
        "pool": pool,
        "provider": pool + "/providers/" + PROVIDER_ID,
        "member": "principalSet://iam.googleapis.com/" + pool + "/attribute.repository_id/" + rid,
        "body": {
            "displayName": "JP Pages GitHub",
            "attributeMapping": {
                "google.subject": "assertion.sub",
                "attribute.repository_id": "assertion.repository_id",
                "attribute.repository_owner_id": "assertion.repository_owner_id",
            },
            "attributeCondition": (
                f"assertion.repository_id == '{rid}' && "
                f"assertion.repository_owner_id == '{oid}' && "
                f"assertion.ref == 'refs/heads/{BRANCH}'"
            ),
            "oidc": {"issuerUri": "https://token.actions.githubusercontent.com"},
        },
    }


def add_identity_binding(policy, member):
    """Retain unrelated bindings, conditions, policy version and etag."""
    policy = copy.deepcopy(policy)
    role = "roles/iam.workloadIdentityUser"
    binding = next((b for b in policy.get("bindings", [])
                    if b.get("role") == role and not b.get("condition")), None)
    if binding is None:
        binding = {"role": role, "members": []}
        policy.setdefault("bindings", []).append(binding)
    if member not in binding["members"]:
        binding["members"].append(member)
    return policy


class API:
    def __init__(self, session, root):
        self.session, self.root = session, root.rstrip("/")

    def response(self, method, path, *, accepted=(200,), **kwargs):
        if not path.startswith("/") or path.startswith("//"):
            raise ConnectionError("API_PATH_INVALID")
        try:
            response = self.session.request(method, self.root + path, timeout=60, **kwargs)
        except Exception:
            raise ConnectionError("API_TRANSPORT_FAILED; rerun this cell to resume") from None
        if response.status_code not in accepted:
            raise ConnectionError(f"SETUP_HTTP_{response.status_code} at {path.split('?')[0]}")
        return response

    def call(self, method, path, **kwargs):
        response = self.response(method, path, **kwargs)
        return response.json() if response.content else {}


def github_api(token):
    import requests
    session = requests.Session()
    session.headers.update({"Authorization": "Bearer " + token,
                            "Accept": "application/vnd.github+json",
                            "X-GitHub-Api-Version": "2022-11-28"})
    return API(session, "https://api.github.com")


def load_registered_config(gh):
    import base64
    import json
    prefix = "/repos/" + REPOSITORY
    repo = gh.call("GET", prefix)
    ref = gh.call("GET", prefix + "/git/ref/heads/" + BRANCH)
    sha = ref["object"]["sha"]

    def read(path):
        data = gh.call("GET", prefix + "/contents/" + path, params={"ref": sha})
        return json.loads(base64.b64decode(data["content"]))

    cfg, registry = read(CONFIG_PATH), read(REGISTRY_PATH)
    lanes = registry.get("active_branches", [])
    if not any(r.get("branch_name") == BRANCH and r.get("allowed_for_work") is True for r in lanes):
        raise ConnectionError("PUBLISHING_BRANCH_NOT_AUTHORIZED")
    access_plan(cfg)  # Validate the complete set before sharing anything.
    return cfg, repo, sha


def permissions(api, fid):
    rows, page = [], None
    while True:
        params = {"fields": "nextPageToken,permissions(id,type,role,emailAddress,deleted)",
                  "supportsAllDrives": "true", "pageSize": 100}
        if page:
            params["pageToken"] = page
        result = api.call("GET", "/drive/v3/files/" + fid + "/permissions", params=params)
        rows.extend(p for p in result.get("permissions", []) if not p.get("deleted"))
        page = result.get("nextPageToken")
        if not page:
            return rows


def sufficient_permission(rows, email, role):
    permitted = {"reader", "writer", "owner", "organizer", "fileOrganizer"}
    if role == "writer":
        permitted.remove("reader")
    return any(p.get("type") == "user" and p.get("emailAddress", "").lower() == email.lower()
               and p.get("role") in permitted for p in rows)


def share_registered_files(api, plan, email, *, apply=False):
    """Preflight every file first. No folder, public, group or email notification."""
    pending, names = [], {}
    for item in plan:
        fid = item["file_id"]
        meta = api.call("GET", "/drive/v3/files/" + fid, params={
            "fields": "id,name,mimeType,capabilities(canShare)", "supportsAllDrives": "true"})
        if meta.get("mimeType") == "application/vnd.google-apps.folder":
            raise ConnectionError("REGISTERED_FOLDER_SHARING_FORBIDDEN")
        names[fid] = meta.get("name", fid)
        existing = permissions(api, fid)
        if sufficient_permission(existing, email, item["role"]):
            continue
        if not meta.get("capabilities", {}).get("canShare"):
            raise ConnectionError("CANNOT_SHARE_REGISTERED_FILE " + fid)
        same_user = [p for p in existing if p.get("type") == "user"
                     and p.get("emailAddress", "").lower() == email.lower()]
        pending.append((item, same_user))
    if apply:
        for item, existing in pending:
            path = "/drive/v3/files/" + item["file_id"] + "/permissions"
            if existing:
                api.call("PATCH", path + "/" + existing[0]["id"],
                         params={"supportsAllDrives": "true"}, json={"role": item["role"]})
            else:
                api.call("POST", path, accepted=(200, 201),
                         params={"sendNotificationEmail": "false", "supportsAllDrives": "true"},
                         json={"type": "user", "role": item["role"], "emailAddress": email})
            if not sufficient_permission(permissions(api, item["file_id"]), email, item["role"]):
                raise ConnectionError("FILE_PERMISSION_READBACK_FAILED " + item["file_id"])
    return {"files": len(plan), "new_or_changed_permissions": len(pending), "names": names}


def wait_operation(api, operation, emit=print):
    deadline = time.monotonic() + 180
    while not operation.get("done"):
        if time.monotonic() > deadline:
            raise ConnectionError("GOOGLE_OPERATION_PENDING; rerun to resume")
        emit("Waiting for Google to finish the identity configuration…")
        time.sleep(5)
        operation = api.call("GET", "/v1/" + operation["name"])
    if operation.get("error"):
        raise ConnectionError("GOOGLE_OPERATION_FAILED_" + str(operation["error"].get("code", "UNKNOWN")))
    return operation.get("response", {})


def ensure_identity(session, project_id, repo, emit=print):
    if not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", project_id):
        raise ConnectionError("ENTER_A_GOOGLE_CLOUD_PROJECT_ID_NOT_ITS_DISPLAY_NAME")
    crm = API(session, "https://cloudresourcemanager.googleapis.com")
    project = crm.call("GET", "/v1/projects/" + project_id)
    if project.get("lifecycleState") != "ACTIVE":
        raise ConnectionError("GOOGLE_CLOUD_PROJECT_NOT_ACTIVE")
    number = str(project["projectNumber"])
    service = API(session, "https://serviceusage.googleapis.com")
    enabled = service.call("POST", f"/v1/projects/{number}/services:batchEnable", json={"serviceIds": [
        "iam.googleapis.com", "iamcredentials.googleapis.com", "sts.googleapis.com",
        "drive.googleapis.com", "sheets.googleapis.com"]})
    wait_operation(service, enabled, emit)
    iam = API(session, "https://iam.googleapis.com")
    email = ACCOUNT_ID + "@" + project_id + ".iam.gserviceaccount.com"
    sa_path = "/v1/projects/" + project_id + "/serviceAccounts/" + email
    response = iam.response("GET", sa_path, accepted=(200, 404))
    if response.status_code == 404:
        iam.call("POST", "/v1/projects/" + project_id + "/serviceAccounts", accepted=(200, 201),
                 json={"accountId": ACCOUNT_ID, "serviceAccount": {"displayName": "JP Pages publishing runner"}})
    spec = federation_spec(number, repo)
    pool_path = "/v1/" + spec["pool"]
    response = iam.response("GET", pool_path, accepted=(200, 404))
    if response.status_code == 404:
        op = iam.call("POST", f"/v1/projects/{number}/locations/global/workloadIdentityPools",
                      params={"workloadIdentityPoolId": POOL_ID},
                      json={"displayName": "JP Pages GitHub", "description": "Exact registered JP publishing branch"})
        wait_operation(iam, op, emit)
    elif response.json().get("state") != "ACTIVE" or response.json().get("disabled"):
        raise ConnectionError("EXISTING_IDENTITY_POOL_DISABLED")
    provider_path = "/v1/" + spec["provider"]
    response = iam.response("GET", provider_path, accepted=(200, 404))
    if response.status_code == 404:
        op = iam.call("POST", pool_path + "/providers", params={"workloadIdentityPoolProviderId": PROVIDER_ID},
                      json=spec["body"])
        wait_operation(iam, op, emit)
    else:
        found = response.json()
        if found.get("disabled") or found.get("state") != "ACTIVE" or any(
                found.get(k) != spec["body"][k] for k in ("attributeMapping", "attributeCondition", "oidc")):
            raise ConnectionError("EXISTING_PROVIDER_DIFFERS; review before replacing any trust policy")
    policy = iam.call("GET", sa_path + ":getIamPolicy", params={"options.requestedPolicyVersion": 3})
    updated = add_identity_binding(policy, spec["member"])
    if updated != policy:
        iam.call("POST", sa_path + ":setIamPolicy", json={"policy": updated})
    verified = iam.call("GET", sa_path + ":getIamPolicy", params={"options.requestedPolicyVersion": 3})
    if add_identity_binding(verified, spec["member"]) != verified:
        raise ConnectionError("IDENTITY_POLICY_READBACK_FAILED")
    return {"provider": spec["provider"], "service_account": email, "project_number": number}


def configure_variables(gh, identity, *, apply=False):
    prefix = "/repos/" + REPOSITORY + "/actions/variables"
    expected = {"JP_GOOGLE_WORKLOAD_IDENTITY_PROVIDER": identity["provider"],
                "JP_GOOGLE_SERVICE_ACCOUNT": identity["service_account"]}
    current = {}
    for name, value in expected.items():
        response = gh.response("GET", prefix + "/" + name, accepted=(200, 404))
        current[name] = response.json().get("value") if response.status_code == 200 else None
        if current[name] and current[name] != value:
            raise ConnectionError("DIFFERENT_EXISTING_GOOGLE_IDENTITY; reuse it or review before replacing")
    if apply:
        for name, value in expected.items():
            if current[name] == value:
                continue
            if current[name] is None:
                gh.call("POST", prefix, accepted=(201,), json={"name": name, "value": value})
            else:
                gh.call("PATCH", prefix + "/" + name, accepted=(204,), json={"name": name, "value": value})
        for name, value in expected.items():
            if gh.call("GET", prefix + "/" + name).get("value") != value:
                raise ConnectionError("GITHUB_VARIABLE_READBACK_FAILED")
    return expected


def dispatch_refresh(gh, expected_sha):
    prefix = "/repos/" + REPOSITORY
    actual = gh.call("GET", prefix + "/git/ref/heads/" + BRANCH)["object"]["sha"]
    if actual != expected_sha:
        raise ConnectionError("PUBLISHING_BRANCH_MOVED; rerun to read the current approved configuration")
    request_id = "colab-" + uuid.uuid4().hex[:16]
    gh.call("POST", prefix + "/actions/workflows/" + WORKFLOW + "/dispatches", accepted=(204,),
            json={"ref": BRANCH, "inputs": {"require_live_sheets": True, "refresh_request_id": request_id}})
    return request_id


def find_refresh_run(gh, request_id, expected_sha):
    data = gh.call("GET", "/repos/" + REPOSITORY + "/actions/workflows/" + WORKFLOW + "/runs",
                   params={"event": "workflow_dispatch", "branch": BRANCH, "per_page": 50})
    matches = [r for r in data.get("workflow_runs", [])
               if r.get("display_title") == "Sheets refresh " + request_id
               and r.get("head_sha") == expected_sha and r.get("head_branch") == BRANCH]
    if len(matches) > 1:
        raise ConnectionError("AMBIGUOUS_REFRESH_RUN")
    return matches[0] if matches else None


def verify_public_receipt(bundle, run_id, sha):
    receipt = bundle.get("publication_receipt", {})
    if (bundle.get("live_google_sheets_fetch") is not True
            or receipt.get("workflow_run_id") != str(run_id) or receipt.get("commit_sha") != sha):
        raise ConnectionError("LIVE_PUBLICATION_NOT_YET_VERIFIED")
    return receipt


def wait_for_refresh(gh, request_id, sha, *, minutes=25, emit=print, public_session=None):
    import requests
    public_session = public_session or requests.Session()
    deadline, last = time.monotonic() + minutes * 60, None
    run = None
    while time.monotonic() < deadline:
        run = find_refresh_run(gh, request_id, sha)
        status = run.get("status") if run else "waiting for GitHub to queue the run"
        if status != last:
            emit("Actions: " + status)
            if run:
                emit(run["html_url"])
            last = status
        if run and run["status"] == "completed":
            if run.get("conclusion") != "success":
                raise ConnectionError("LIVE_REFRESH_" + str(run.get("conclusion", "UNKNOWN")).upper()
                                      + "; inspect " + run["html_url"])
            try:
                response = public_session.get(SITE_ROOT + "ops/data/map-data.v1.json",
                    params={"jp_run": run["id"]}, headers={"Cache-Control": "no-cache"}, timeout=30)
                if response.ok:
                    receipt = verify_public_receipt(response.json(), run["id"], sha)
                    return {"verified": True, "run_url": run["html_url"], "receipt": receipt}
            except ConnectionError:
                pass  # Pages/CDN propagation is not a successful live readback yet.
            except Exception:
                pass
        time.sleep(15)
    link = run.get("html_url", SITE_ROOT) if run else "https://github.com/" + REPOSITORY + "/actions"
    raise ConnectionError("VERIFICATION_PENDING; no success claimed. Check " + link)


def connect(session, token, project_id, *, emit=print, wait_minutes=25):
    gh = github_api(token)
    cfg, repo, sha = load_registered_config(gh)
    plan = access_plan(cfg)
    email = ACCOUNT_ID + "@" + project_id + ".iam.gserviceaccount.com"
    drive = API(session, "https://www.googleapis.com")
    # Check all sharing rights before IAM changes, then reject any conflicting
    # existing repository identity before enabling or creating Cloud resources.
    preview = share_registered_files(drive, plan, email)
    project = API(session, "https://cloudresourcemanager.googleapis.com").call("GET", "/v1/projects/" + project_id)
    spec = federation_spec(project["projectNumber"], repo)
    configure_variables(gh, {"provider": spec["provider"], "service_account": email})
    emit(f"Reusing {len(plan)} registered files; {sum(x['role'] == 'writer' for x in plan)} output writers. No Sheet creation.")
    identity = ensure_identity(session, project_id, repo, emit)
    share_registered_files(drive, plan, email, apply=True)
    configure_variables(gh, identity, apply=True)
    emit("Identity and exact file permissions verified. IAM propagation can take several minutes.")
    # Keep this wait visible and shorter than a minute; the next dispatch must
    # actually authenticate and fetch, rather than assuming propagation finished.
    emit("Waiting 45 seconds for initial identity propagation…")
    time.sleep(45)
    request_id = dispatch_refresh(gh, sha)
    result = wait_for_refresh(gh, request_id, sha, minutes=wait_minutes, emit=emit)
    emit("LIVE SHEETS → PUBLISHING SHEET → VALIDATED JSON → ACTIONS → PAGES: VERIFIED")
    return result


## Preview the existing file contract
The preview below is a public configuration snapshot from the already-registered pipeline. Actual setup reloads the current configuration and branch authorization from GitHub before making any changes. It shares individual files, never their parent folders.

In [ ]:
# Step 2 — no account credentials or external writes.
import json
PREVIEW_CONFIG = json.loads('{"version":"V000006","repository":"FabinGurung/JP_Communication-Command-Centre-and-Resource-Hub","publishing_workbook_id":"17y3ki0SLSG_Hmz8AipfWhhffnKastcjHvRqIdepJMu4","previous_versions_folder_id":"12-eWCCG56jJBFlFfHa_49sxgydZAbRCj","execution_owner":"GITHUB_ACTIONS","connection_state":"PENDING_GOOGLE_IDENTITY_AND_FIRST_LIVE_RUN","trigger":"REGISTERED_PAGES_BRANCH_PUSH_OR_MANUAL_WORKFLOW_RUN","schedule_state":"NOT_ENABLED__DEFAULT_BRANCH_WORKFLOW_REQUIRED","sources":{"projects":{"workbook_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"projects"},"organizations":{"workbook_id":"1cwubE_4nB2SAEdLu6ry80G0HeNjbBdu4X3DA6jrcQ1E","legacy_id":"1cwubE_4nB2SAEdLu6ry80G0HeNjbBdu4X3DA6jrcQ1E","tab":"organizations"},"people":{"workbook_id":"1gBYUKwt3_eUy-h9FdDCv4o2-RxxT6dBqqY4yRG2Hws4","legacy_id":"1gBYUKwt3_eUy-h9FdDCv4o2-RxxT6dBqqY4yRG2Hws4","tab":"people"},"project_parties":{"workbook_id":"1SC0Iz5IKwUCx5elSVyok0Z0hV0Z5hHiMCzkWWv0Y2uU","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_parties"},"project_funding":{"workbook_id":"1pNXK4JQnYI62V8iZsI64yyVEYiRxB3UZ7Z9R6JxoZUs","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_funding"},"work_catalog":{"workbook_id":"1D4T08NciyhlMxxTgTtik_SJWf-UaIytB56fNRsvuqms","legacy_id":"1D4T08NciyhlMxxTgTtik_SJWf-UaIytB56fNRsvuqms","tab":"work_catalog"},"project_work_items":{"workbook_id":"17YPRpaXIdHloH5-lzsjUpNvXLbsDfrIrlsn2jZ-7KKc","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_work_items"},"materials_master":{"workbook_id":"10D96YqZn6NbbSm6X6nsPVw4tG_IELIj957uZ2DmhJZs","legacy_id":"1D4T08NciyhlMxxTgTtik_SJWf-UaIytB56fNRsvuqms","tab":"materials_master"},"project_material_inventory":{"workbook_id":"1J0c-usE3jUubT_hCTxxklHzyVBzxoFZaJtCsuG3toWA","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_material_inventory","read_mode":"EMPTY_TEMPLATE_NOT_CONNECTED"},"project_components":{"workbook_id":"1kAtyZPQOdSzxDS83AYwIBzrj5OIsvBKE_8orH8ez07U","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_components"},"project_source_links":{"workbook_id":"1uC1IZG440Hrg78hurYlmVA_OJG5Z-04TgKcxs1_TZmU","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_source_links"},"project_opportunities":{"workbook_id":"1dAz0QhEj_UItB-m_BIt7jJZ_qJvjmJ8Ou9rVv2yXqWE","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_opportunities"},"organization_locations":{"workbook_id":"12wGg0UBLbN03u3EhJN-uQV9O2s_FRVwX9V_D2rTDXC8","legacy_id":"1cwubE_4nB2SAEdLu6ry80G0HeNjbBdu4X3DA6jrcQ1E","tab":"organization_locations"},"project_publication_settings":{"workbook_id":"1tDpsphzARq-rk1MExU4DXC8mG_clh-0TXJwRr2qljXU","legacy_id":"1JqzjBNBCT_AAIBkuGsIR5VBOJ3_HyifdPDc7K_jF5Bo","tab":"project_publication_settings"}},"refresh_fields":{"projects":["project_name","display_name","location_ward_number","latitude","longitude","status","last_updated"],"organizations":["organization_name","active"],"work_catalog":["work_code","description","category","subcategory","default_unit","active"],"materials_master":["material_code","material_name","material_category","subcategory","default_unit","active"],"organization_locations":["location_name","location_type","location_ward_number","latitude","longitude"],"project_opportunities":["opportunity_name","location_ward_number","status","latitude","longitude"],"project_components":["component_type","component_name","sequence_no","latitude","longitude","status"],"project_publication_settings":["show_project_name","show_coordinates","show_status","show_ward_number","show_project_sector","show_implementation_method","show_legal_contractor","show_executing_company_role","show_general_work_summary"]},"public_admission":"Existing admitted record IDs and explicit refresh fields only; private and newly discovered rows stay out. Free text, money, measured progress, evidence and previously redacted fields retain their reviewed values pending admission.","project_visibility_fields":{"show_project_name":["project_name","display_name"],"show_coordinates":["latitude","longitude"],"show_status":["status"],"show_ward_number":["location_ward_number"],"show_project_sector":["project_sector"],"show_implementation_method":["implementation_mechanism"],"show_legal_contractor":["executing_company_id"]},"previous_versions_file_id":"1c7TD0CyXCJE1Z_Ks5KHje35P0y_VDEQF","source_ownership_state":"14_DISTINCT_OWNERS__INITIAL_API_ASSEMBLY_VERIFIED","initial_assembly_date_npt":"2026-10-26","resources_source":{"workbook_id":"1e019Jac4lDveZ2glXMph4RLHivqPFEachbcNi-R1DZk","tab":"project_resources","pk":"resource_id"},"notification_sources":{"message_sources":{"workbook_id":"1aWHzjEK0doiUcBbqhySiA6DWCp8d-aq2I08McslilAQ","tab":"message_sources"},"site_notifications":{"workbook_id":"1ZPtTos8t9Cym_qnpiUhiyuTxndSc6sGzzv390ahAw4I","tab":"site_notifications"}},"notification_policy":{"source_sheet_url":"https://docs.google.com/spreadsheets/d/1ZPtTos8t9Cym_qnpiUhiyuTxndSc6sGzzv390ahAw4I/edit?usp=drivesdk","admitted_sources":{"MSG-SLACK-C0C2LFV6XLM-1792979252137289":{"version":"V000001","sha256":"de933cd9dc116707c0e62767c629d2fa119d34209f7c66059fdbfc7b8df3d52c","source_url":"https://jp09.slack.com/archives/C0C2LFV6XLM/p1792979252137289","drive_source_url":"https://drive.google.com/file/d/1nMkMVfLI8JW96TH9XZCYqEjr63wxZmI_/view?usp=drivesdk","project_id":"PRJ-000020"}}},"field_sources":[{"workbook_id":"15szptLz7q3-Rote1SnNuIqdcrxrm0z2m0k7yx9j51c4","tab":"2_PEOPLE_DIRECTORY","adapter":"contacts","table":"company_contacts","expected_header":["name","phone","role","project_s","verification","notes","contact_id"],"company":"Fishtail Builders","person":""},{"workbook_id":"1Cyuj0uti-IN7ddDNVdAd1kM0NNpATdTnNabfcY7P1cw","tab":"2_PEOPLE_DIRECTORY","adapter":"contacts","table":"company_contacts","expected_header":["name","phone","role","project_s","verification","notes","contact_id"],"company":"Rohini Engineering and Builders","person":""},{"workbook_id":"1YshLrDAJDg3CO8F1jmIr2Eucy0-TKwiviVNF5tiAfT0","tab":"materials_inventory","adapter":"materials","table":"material_movements","expected_header":["movement_id","date","project_id","site","material","movement","quantity","unit","remarks"],"company":"Fishtail Builders","person":""},{"workbook_id":"1GQQDRg3bDPVIrfs1oVm_ycmyBE5NlpxEKzu1DsAz6tk","tab":"materials_inventory","adapter":"materials","table":"material_movements","expected_header":["movement_id","date","project_id","site","material","movement","quantity","unit","remarks"],"company":"Rohini Engineering and Builders","person":""},{"workbook_id":"1-pE1d4uxytIZsKF8Tzu3eBDlXL4MLIYiGGvVoKMqo1k","tab":"2_OPS_DAILY_PROGRESS","adapter":"progress","table":"site_progress","expected_header":["date","site","status","work_location","qty_note","progress_event_id_pk","project_id_fk","source_artifact_id"],"company":"Fishtail Builders","person":""},{"workbook_id":"1-pE1d4uxytIZsKF8Tzu3eBDlXL4MLIYiGGvVoKMqo1k","tab":"3_OPS_DAILY_READINESS","adapter":"readiness","table":"site_readiness","expected_header":["date","site","work_id","activity","location","materials","manpower","ppe","ppe_responsibility","work_briefing","tools","preconditions"],"company":"Fishtail Builders","person":""},{"workbook_id":"1agDIiVfTwicWAxix07HaukX7EemNdO4IyghYcBH_kys","tab":"2_OPS_DAILY_PROGRESS","adapter":"progress","table":"site_progress","expected_header":["date","site","status","work_location","qty_note","progress_event_id_pk","project_id_fk","source_artifact_id"],"company":"Rohini Engineering and Builders","person":""},{"workbook_id":"1agDIiVfTwicWAxix07HaukX7EemNdO4IyghYcBH_kys","tab":"3_OPS_DAILY_READINESS","adapter":"readiness","table":"site_readiness","expected_header":["date","site","work_id","activity","location","materials","manpower","ppe","ppe_responsibility","work_briefing","tools","preconditions"],"company":"Rohini Engineering and Builders","person":""},{"workbook_id":"1blVxu1qfmZU06J5QNjwpyOBDnqtR9HzC_GEzk5WWFBw","tab":"Attendance","adapter":"attendance_split","table":"site_attendance","expected_header":["attendance_date","status","","visit_date","site_visited","note"],"company":"Fishtail Builders","person":"Paras Pariyar"},{"workbook_id":"16B3tBDNF6YQgt79tptMSygSrsUZXoTWI3yDzUmNnWuc","tab":"Attendance","adapter":"attendance_split","table":"site_attendance","expected_header":["attendance_date","status","","visit_date","site_visited","note"],"company":"Fishtail Builders","person":"Shiv Narayan Chaudhary"},{"workbook_id":"11Dpvqm_joE4ZnySYSkGObMIya-ILHRdZf4GC3ZlRoj0","tab":"2_Attendance","adapter":"attendance_checkbox","table":"site_attendance","expected_header":["date","14_bishal_paija"],"company":"Rohini Engineering and Builders","person":"Krishna Kumar Gupta"},{"workbook_id":"1blVxu1qfmZU06J5QNjwpyOBDnqtR9HzC_GEzk5WWFBw","tab":"Expenses","adapter":"expenses_rich","table":"site_expenses","expected_header":["id","date","item","project_name","amount","hardware","bill_photo","remarks"],"company":"Fishtail Builders","person":"Paras Pariyar"},{"workbook_id":"11Dpvqm_joE4ZnySYSkGObMIya-ILHRdZf4GC3ZlRoj0","tab":"3_Expenses","adapter":"expenses_simple","table":"site_expenses","expected_header":["date","project","amount_npr","remarks"],"company":"Rohini Engineering and Builders","person":"Krishna Kumar Gupta"},{"workbook_id":"1S1fkWzo6XGzyD3p92_pTG32zZ9fF3YICAv3HPTomZsE","tab":"KKG_VENDOR_SUMMARY_AUTO","adapter":"vendor_summary","table":"vendor_payments","expected_header":["id","vendor_purpose","amount_npr","qr_npr","discount_npr"],"company":"Rohini Engineering and Builders","person":""},{"workbook_id":"1noepsgHiW3MM8mBKaF2BzbgLK1WsMsbssrVNS5_ucx0","tab":"01_HUMAN_REVIEW","adapter":"supplier_review","table":"source_review_queue","expected_header":["photo_file_preview","human_input_transcription","document_type","bill_no","bill_date","total_amount","payment_status","review_status"],"company":"Fishtail Builders","person":""},{"workbook_id":"1NvvOD7UUDAvWVnGOC33q2MOy3h6ZDoD1Y2es9l3dpiY","tab":"MEASUREMENT_RECORDS","adapter":"measurement_review","table":"source_review_queue","expected_header":["record_id","project_id","project_name","work_category","work_item","measurement_type","description","quantity","unit","rate","amount","measurement_date"],"company":"Fishtail Builders","person":""}],"contacts_reviewed_projection":"ops/data/company-contacts-reviewed.v1.json","field_publication_policy":"Eight derived private joined tables. Only previously admitted contact identities and approved contact fields enter the public bundle. Attendance, expenses, payments, raw messages, evidence and unreviewed measurement quantities remain private.","field_company_names":{"ORG-000001":"Fishtail Builders","ORG-000002":"Rohini Engineering and Builders"}}')
preview = access_plan(PREVIEW_CONFIG)
print("Registered files:", len(preview))
print("Readers:", sum(x['role'] == 'reader' for x in preview))
print("Writers:", sum(x['role'] == 'writer' for x in preview))
print("No new source workbook. Empty templates excluded. No folder-wide sharing.")
for item in preview:
    print(item['role'].upper(), ' | '.join(item['purposes']), ' | ', item['file_id'])


## Connect and verify
This cell is the only step that changes permissions or Cloud/GitHub settings. If a different identity is already configured, it stops for review instead of overwriting it. If a required file cannot be shared, it identifies that file ID. A 403 usually means the signed-in account lacks the required permission; it does not mean the source must be copied.

The run can take 10–25 minutes because the existing production workflow validates the repository before deploying Pages. Keep the notebook open. You can follow the printed Actions link while it runs.

In [ ]:
# Step 3 — official Google consent, hidden GitHub token, then exact run verification.
if not CONNECT_ACTIONS:
    print("PREVIEW ONLY — Google identity and live publication have NOT been configured or tested.")
else:
    from google.colab import auth
    import google.auth
    from google.auth.transport.requests import AuthorizedSession
    from getpass import getpass
    if not PROJECT_ID.strip():
        print("Create/select your Google Cloud project: https://console.cloud.google.com/projectcreate")
        PROJECT_ID = input("Google Cloud project ID: ").strip()
    if not re.fullmatch(r"[a-z][a-z0-9-]{4,28}[a-z0-9]", PROJECT_ID.strip()):
        raise ValueError("Enter the project ID, e.g. jp-pages-123456, then rerun this cell.")
    auth.authenticate_user()
    credentials, _ = google.auth.default(scopes=[
        "https://www.googleapis.com/auth/cloud-platform",
        "https://www.googleapis.com/auth/drive",
        "https://www.googleapis.com/auth/spreadsheets"])
    google_session = AuthorizedSession(credentials)
    github_token = getpass("Short-lived GitHub setup token (hidden): ").strip()
    if not github_token:
        raise ValueError("No token entered; no setup attempted.")
    try:
        result = connect(google_session, github_token, PROJECT_ID.strip(), wait_minutes=WAIT_MINUTES)
        print("Verified run:", result['run_url'])
        print("Public receipt:", result['receipt'])
    except ConnectionError as err:
        print(str(err))
        print("NOT COMPLETE. Fix the stated permission/configuration issue, then rerun this cell. Existing files and IDs are reused.")
    finally:
        github_token = None
        google_session.close()
        credentials = None


## Next steps

- Check the native **Website Publishing and Source Directory → Start Here** for live Sheet links. Humans edit those sources; the publishing workbook is generated.
- Use [Actions](https://github.com/FabinGurung/JP_Communication-Command-Centre-and-Resource-Hub/actions/workflows/daily-ops-branch-preview.yml) for later required-live refreshes. The hourly dispatcher, if merged on the default branch, runs this same feature-branch engine.
- Download Excel directly from the source Sheet or publishing Sheet; the Sheet remains its editing authority.
- Revoke the short-lived GitHub setup token after success; routine Actions uses keyless Google federation and GitHub's own workflow token.
- Clear executed authentication outputs before saving a shared copy. Do not create another live notebook or source Sheet to record success. Record the actual run URL and receipt in the existing manifest/history instead.